# Direct signal classification — leakage-safe dataset

Этап подготавливает causal dataset для прямого предсказания `good_day` и `window_closing`. Модели не обучаются. Golden methodology является source of truth для targets, metrics methodology — для будущей оценки.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
golden_methodology_path = ROOT / 'docs/golden_label_methodology.md'
metrics_methodology_path = ROOT / 'docs/metrics_methodology.md'
for path in (golden_methodology_path, metrics_methodology_path):
    if not path.exists(): raise FileNotFoundError(path)
golden_methodology = golden_methodology_path.read_text(encoding='utf-8')
metrics_methodology = metrics_methodology_path.read_text(encoding='utf-8')
print('TARGET METHODOLOGY:')
print('good_day = rate_T <= min(rate[T-10:T+10]) * (1 + 100/10000); complete 21-calendar-day window.')
print('window_closing = good=0 AND rebound above prior-10-calendar-day minimum in (100, 200] bps AND next-10-calendar-day median rise >=100 bps.')
print('\nMETRICS METHODOLOGY:')
print('precision = required and declared primary; explicit formula is absent.')
print('F0.5 = required; explicit formula is absent.')
print('hit_rate = generic definition is absent; safety_hit_rate and closing_confirmation_hit are defined separately.')
print('uplift = classification lift is precision over target prevalence, but an explicit formula/name is absent; safety_lift is signal safety hit rate / random safety hit rate; economic ratio lift is prohibited.')
print('random baseline = policy-matched schedules with same corridor, period, push count and eligible dates, four-calendar-day cooldown and max two pushes per ISO week; report mean and 5–95 percentiles plus one-sided randomization p-value.')
print('aggregation/micro/macro = definitions are absent; methodology requires reporting separately by scenario, corridor, horizon and full portfolio.')

TARGET METHODOLOGY:
good_day = rate_T <= min(rate[T-10:T+10]) * (1 + 100/10000); complete 21-calendar-day window.
window_closing = good=0 AND rebound above prior-10-calendar-day minimum in (100, 200] bps AND next-10-calendar-day median rise >=100 bps.

METRICS METHODOLOGY:
precision = required and declared primary; explicit formula is absent.
F0.5 = required; explicit formula is absent.
hit_rate = generic definition is absent; safety_hit_rate and closing_confirmation_hit are defined separately.
uplift = classification lift is precision over target prevalence, but an explicit formula/name is absent; safety_lift is signal safety hit rate / random safety hit rate; economic ratio lift is prohibited.
random baseline = policy-matched schedules with same corridor, period, push count and eligible dates, four-calendar-day cooldown and max two pushes per ISO week; report mean and 5–95 percentiles plus one-sided randomization p-value.
aggregation/micro/macro = definitions are absent; methodology 

## Exact targets and causal source

Exact columns in `golden_labels.parquet` are `good` and `closing`; они переименовываются в `y_good_day` и `y_window_closing`. Causal features переиспользуются из Stage 26 prepared dataset. Его continuous future target удаляется до merge и не становится feature.

In [2]:
golden_path = ROOT / 'data/labels/golden_labels.parquet'
causal_path = ROOT / 'reports/direct_deterioration_dataset.parquet'
for path in (golden_path, causal_path):
    if not path.exists(): raise FileNotFoundError(path)
golden = pd.read_parquet(golden_path)
causal = pd.read_parquet(causal_path)
for frame in (golden, causal): frame['date'] = pd.to_datetime(frame.date, errors='raise')
required_targets = {'corridor', 'date', 'good', 'closing', 'label_horizon_days', 'closing_horizon_days'}
if missing := required_targets.difference(golden.columns): raise ValueError(f'Missing target columns: {sorted(missing)}')
if not golden.label_horizon_days.eq(10).all() or not golden.closing_horizon_days.eq(10).all():
    raise ValueError('Unexpected target horizon; purge must follow methodology')
future_source_columns = [c for c in causal.columns if any(token in c.lower() for token in ('future', 'actual', 'golden', 'label', 'regret', 'centered'))]
expected_stage26_target = 'future_median_deterioration_10_bps'
assert future_source_columns == [expected_stage26_target], future_source_columns
causal_inputs = causal.drop(columns=[expected_stage26_target, 'split'])
targets = golden[['corridor', 'date', 'good', 'closing']].rename(columns={'good': 'y_good_day', 'closing': 'y_window_closing'})
dataset = causal_inputs.merge(targets, on=['corridor', 'date'], how='inner', validate='one_to_one')
dataset['y_good_day'] = dataset.y_good_day.astype('int8')
dataset['y_window_closing'] = dataset.y_window_closing.astype('int8')
print('exact target columns: good -> y_good_day; closing -> y_window_closing')
print('rows before temporal purge:', len(dataset))

exact target columns: good -> y_good_day; closing -> y_window_closing
rows before temporal purge: 9460


In [3]:
PURGE_CALENDAR_DAYS = 10
conditions = [
    dataset.date.le(pd.Timestamp('2024-12-31') - pd.Timedelta(days=PURGE_CALENDAR_DAYS)),
    dataset.date.between(pd.Timestamp('2025-01-01'), pd.Timestamp('2025-12-31') - pd.Timedelta(days=PURGE_CALENDAR_DAYS)),
    dataset.date.dt.year.eq(2026),
]
dataset['split'] = np.select(conditions, ['TRAIN', 'VALIDATION', 'TEST'], default='PURGED')
purged_rows = dataset.loc[dataset.split.eq('PURGED'), ['corridor', 'date']].copy()
dataset = dataset.loc[dataset.split.ne('PURGED')].sort_values(['corridor', 'date']).reset_index(drop=True)
print('purge calendar days:', PURGE_CALENDAR_DAYS)
print('purged rows:', len(purged_rows), 'range:', purged_rows.date.min().date(), '—', purged_rows.date.max().date())

purge calendar days: 10
purged rows: 75 range: 2024-12-23 — 2025-12-31


In [4]:
metadata_columns = {'corridor', 'date', 'rate_t', 'split'}
target_columns = {'y_good_day', 'y_window_closing'}
feature_columns = [c for c in dataset.columns if c not in metadata_columns | target_columns]
forbidden_tokens = ('future', 'actual', 'golden', 'label', 'regret', 'centered', 'lead')
forbidden_features = [c for c in feature_columns if any(token in c.lower() for token in forbidden_tokens)]
negative_shift_features = [c for c in feature_columns if 'shift(-' in c.lower()]
split_summary = dataset.groupby('split').agg(
    first_date=('date', 'min'), last_date=('date', 'max'), rows=('date', 'size'),
    good_positives=('y_good_day', 'sum'), closing_positives=('y_window_closing', 'sum'),
).reindex(['TRAIN', 'VALIDATION', 'TEST'])
split_summary['good_positive_rate'] = split_summary.good_positives / split_summary.rows
split_summary['closing_positive_rate'] = split_summary.closing_positives / split_summary.rows
corridor_summary = dataset.groupby(['split', 'corridor']).agg(
    rows=('date', 'size'), good_positives=('y_good_day', 'sum'), closing_positives=('y_window_closing', 'sum'),
).reset_index()
audit = {
    'five corridors in every split': dataset.groupby('split').corridor.nunique().eq(5).all(),
    'no missing values': not dataset.isna().any().any(),
    'unique corridor/date': not dataset.duplicated(['corridor', 'date']).any(),
    'binary targets': dataset.y_good_day.isin([0, 1]).all() and dataset.y_window_closing.isin([0, 1]).all(),
    'targets excluded from features': target_columns.isdisjoint(feature_columns),
    'no forbidden feature names': len(forbidden_features) == 0 and len(negative_shift_features) == 0,
    'train labels do not cross validation boundary': dataset.loc[dataset.split.eq('TRAIN'), 'date'].max() + pd.Timedelta(days=10) < pd.Timestamp('2025-01-01'),
    'validation labels do not cross test boundary': dataset.loc[dataset.split.eq('VALIDATION'), 'date'].max() + pd.Timedelta(days=10) < pd.Timestamp('2026-01-01'),
    'test is locked 2026': dataset.loc[dataset.split.eq('TEST'), 'date'].dt.year.eq(2026).all(),
}
leakage_pass = all(audit.values())
display(split_summary)
display(corridor_summary)
print('corridors:', sorted(dataset.corridor.unique()))
print('FEATURE COUNT:', len(feature_columns))
print('missing:', int(dataset.isna().sum().sum()), 'duplicates:', int(dataset.duplicated(['corridor', 'date']).sum()))
display(pd.DataFrame(audit.items(), columns=['check', 'passed']))

,first_date,last_date,rows,good_positives,closing_positives,good_positive_rate,closing_positive_rate
split,,,,,,,
TRAIN,2019-05-24,2024-12-20,7280,2232,380,0.306593,0.052198
VALIDATION,2025-01-01,2025-12-19,1265,336,50,0.265613,0.039526
TEST,2026-01-01,2026-08-24,840,203,52,0.241667,0.061905


,split,corridor,rows,good_positives,closing_positives
0,TEST,AMD_RUB,168,44,11
1,TEST,KGS_RUB,168,46,8
2,TEST,KZT_RUB,168,24,13
3,TEST,TJS_RUB,168,51,10
4,TEST,UZS_RUB,168,38,10
5,TRAIN,AMD_RUB,1456,399,83
6,TRAIN,KGS_RUB,1456,422,78
7,TRAIN,KZT_RUB,1456,518,71
8,TRAIN,TJS_RUB,1456,456,79
9,TRAIN,UZS_RUB,1456,437,69


corridors: ['AMD_RUB', 'KGS_RUB', 'KZT_RUB', 'TJS_RUB', 'UZS_RUB']
FEATURE COUNT: 36
missing: 0 duplicates: 0


,check,passed
0,five corridors in every split,True
1,no missing values,True
2,unique corridor/date,True
3,binary targets,True
4,targets excluded from features,True
5,no forbidden feature names,True
6,train labels do not cross validation boundary,True
7,validation labels do not cross test boundary,True
8,test is locked 2026,True


In [5]:
output_path = ROOT / 'reports/direct_signal_ml_dataset.parquet'
save_columns = ['corridor', 'date', 'rate_t', *feature_columns, 'y_good_day', 'y_window_closing', 'split']
dataset[save_columns].to_parquet(output_path, index=False)
saved = pd.read_parquet(output_path)
assert len(saved) == len(dataset) and not saved.isna().any().any()
print('saved:', output_path.relative_to(ROOT), 'rows:', len(saved))
print('TRAIN PERIOD:', split_summary.loc['TRAIN', 'first_date'].date(), '—', split_summary.loc['TRAIN', 'last_date'].date())
print('VALIDATION PERIOD:', split_summary.loc['VALIDATION', 'first_date'].date(), '—', split_summary.loc['VALIDATION', 'last_date'].date())
print('TEST PERIOD:', split_summary.loc['TEST', 'first_date'].date(), '—', split_summary.loc['TEST', 'last_date'].date())
print('FEATURE COUNT:', len(feature_columns))
print('TARGET COUNTS:')
print(split_summary[['rows', 'good_positives', 'closing_positives']].to_string())
print('LEAKAGE PRECHECK:', 'PASS' if leakage_pass else 'FAIL')

saved: reports\direct_signal_ml_dataset.parquet rows: 9385
TRAIN PERIOD: 2019-05-24 — 2024-12-20
VALIDATION PERIOD: 2025-01-01 — 2025-12-19
TEST PERIOD: 2026-01-01 — 2026-08-24
FEATURE COUNT: 36
TARGET COUNTS:
            rows  good_positives  closing_positives
split                                              
TRAIN       7280            2232                380
VALIDATION  1265             336                 50
TEST         840             203                 52
LEAKAGE PRECHECK: PASS


## TRAIN-only binary model comparison

Для каждого из двух targets обучаются отдельные модели, а внутри model family — отдельная модель на corridor. Используется по одной фиксированной конфигурации без hyperparameter search. Class imbalance компенсируется `class_weight='balanced'` или CatBoost `auto_class_weights='Balanced'`. TEST не используется и не оценивается.

Classification diagnostics при threshold 0.5 рассчитываются только публичной функцией `src.backtest.metrics.classification_metrics` — source of truth из metrics methodology. Её `average_precision` показывается как PR-AUC/average precision. ROC-AUC является отдельно запрошенной дополнительной ranking metric.

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
from src.backtest.metrics import classification_metrics

ml_dataset = pd.read_parquet(ROOT / 'reports/direct_signal_ml_dataset.parquet')
ml_dataset['date'] = pd.to_datetime(ml_dataset.date, errors='raise')
train = ml_dataset.loc[ml_dataset.split.eq('TRAIN')].copy()
validation = ml_dataset.loc[ml_dataset.split.eq('VALIDATION')].copy()
assert train.date.max() <= pd.Timestamp('2024-12-31')
assert validation.date.dt.year.eq(2025).all()
MODEL_TARGETS = {'GOOD_DAY': 'y_good_day', 'WINDOW_CLOSING': 'y_window_closing'}
MODEL_NON_FEATURES = {'corridor', 'date', 'rate_t', 'split', *MODEL_TARGETS.values()}
ml_features = [c for c in ml_dataset.columns if c not in MODEL_NON_FEATURES]
assert not train[ml_features].isna().any().any() and not validation[ml_features].isna().any().any()

def make_models():
    return {
        'LOGISTIC_REGRESSION': Pipeline([
            ('scale', StandardScaler()),
            ('model', LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000, solver='lbfgs', random_state=42)),
        ]),
        'HIST_GRADIENT_BOOSTING': HistGradientBoostingClassifier(
            learning_rate=0.05, max_iter=200, max_leaf_nodes=15, min_samples_leaf=20,
            l2_regularization=0.1, class_weight='balanced', early_stopping=False, random_state=42,
        ),
        'CATBOOST': CatBoostClassifier(
            iterations=300, learning_rate=0.05, depth=6, loss_function='Logloss',
            auto_class_weights='Balanced', random_seed=42, verbose=False,
            allow_writing_files=False, thread_count=-1,
        ),
    }

prediction_parts = []
for target_name, target_column in MODEL_TARGETS.items():
    for corridor in sorted(validation.corridor.unique()):
        corridor_train = train.loc[train.corridor.eq(corridor)]
        corridor_validation = validation.loc[validation.corridor.eq(corridor)]
        if corridor_train.empty or corridor_validation.empty: raise ValueError(f'Empty split for {corridor}')
        if corridor_train[target_column].nunique() != 2: raise ValueError(f'Non-binary train coverage for {corridor}/{target_name}')
        for model_name, model in make_models().items():
            model.fit(corridor_train[ml_features], corridor_train[target_column])
            probability = model.predict_proba(corridor_validation[ml_features])[:, 1]
            prediction_parts.append(pd.DataFrame({
                'corridor': corridor_validation.corridor.to_numpy(),
                'date': corridor_validation.date.to_numpy(),
                'model': model_name,
                'target': target_name,
                'actual': corridor_validation[target_column].astype('int8').to_numpy(),
                'predicted_probability': probability.astype(float),
                'diagnostic_prediction_0_5': (probability >= 0.5).astype('int8'),
                'split': 'VALIDATION',
                'train_end': corridor_train.date.max(),
            }))
validation_predictions = pd.concat(prediction_parts, ignore_index=True).sort_values(['target', 'model', 'corridor', 'date']).reset_index(drop=True)
assert validation_predictions.train_end.le(pd.Timestamp('2024-12-31')).all()
assert validation_predictions.date.dt.year.eq(2025).all()
print('prediction rows:', len(validation_predictions))
display(validation_predictions.groupby(['target', 'model']).size().rename('rows').reset_index())

prediction rows: 7590


,target,model,rows
0,GOOD_DAY,CATBOOST,1265
1,GOOD_DAY,HIST_GRADIENT_BOOSTING,1265
2,GOOD_DAY,LOGISTIC_REGRESSION,1265
3,WINDOW_CLOSING,CATBOOST,1265
4,WINDOW_CLOSING,HIST_GRADIENT_BOOSTING,1265
5,WINDOW_CLOSING,LOGISTIC_REGRESSION,1265


In [7]:
metric_rows = []
for (target_name, model_name), group in validation_predictions.groupby(['target', 'model'], sort=True):
    metrics = classification_metrics(
        group.actual, group.diagnostic_prediction_0_5,
        y_score=group.predicted_probability, beta=0.5,
    )
    metric_rows.append({
        'target': target_name, 'model': model_name, 'threshold': 0.5,
        'PR_AUC_average_precision': metrics['average_precision'],
        'ROC_AUC': roc_auc_score(group.actual, group.predicted_probability),
        **{key: metrics[key] for key in [
            'rows', 'true_positive', 'false_positive', 'false_negative', 'true_negative',
            'predicted_positive', 'actual_positive', 'precision', 'recall', 'f_beta',
            'beta', 'prevalence', 'random_precision', 'classification_lift',
        ]},
    })
validation_metrics = pd.DataFrame(metric_rows).sort_values(['target', 'model']).reset_index(drop=True)
compact_columns = [
    'target', 'model', 'PR_AUC_average_precision', 'ROC_AUC', 'precision', 'recall',
    'f_beta', 'predicted_positive', 'actual_positive', 'prevalence', 'classification_lift',
]
print('VALIDATION 2025 — THRESHOLD 0.5 DIAGNOSTIC')
display(validation_metrics[compact_columns])

VALIDATION 2025 — THRESHOLD 0.5 DIAGNOSTIC


,target,model,PR_AUC_average_precision,ROC_AUC,precision,recall,f_beta,predicted_positive,actual_positive,prevalence,classification_lift
0,GOOD_DAY,CATBOOST,0.468161,0.757711,0.476190,0.535714,0.487013,378,336,0.265613,1.792800
1,GOOD_DAY,HIST_GRADIENT_BOOSTING,0.438726,0.754793,0.476309,0.568452,0.492268,401,336,0.265613,1.793248
2,GOOD_DAY,LOGISTIC_REGRESSION,0.443292,0.747754,0.461832,0.720238,0.497533,524,336,0.265613,1.738743
3,WINDOW_CLOSING,CATBOOST,0.150742,0.777070,0.181818,0.120000,0.164835,33,50,0.039526,4.600000
4,WINDOW_CLOSING,HIST_GRADIENT_BOOSTING,0.166048,0.785926,0.305556,0.220000,0.283505,36,50,0.039526,7.730556
5,WINDOW_CLOSING,LOGISTIC_REGRESSION,0.063314,0.597646,0.063218,0.440000,0.076283,348,50,0.039526,1.599425


In [8]:
prediction_path = ROOT / 'reports/direct_signal_ml_validation_predictions.parquet'
validation_predictions.to_parquet(prediction_path, index=False)
saved_predictions = pd.read_parquet(prediction_path)
assert len(saved_predictions) == len(validation_predictions)
assert saved_predictions.split.eq('VALIDATION').all()
assert saved_predictions.date.dt.year.eq(2025).all()
print('COMPACT MODEL TABLE — GOOD_DAY')
display(validation_metrics.loc[validation_metrics.target.eq('GOOD_DAY'), compact_columns])
print('COMPACT MODEL TABLE — WINDOW_CLOSING')
display(validation_metrics.loc[validation_metrics.target.eq('WINDOW_CLOSING'), compact_columns])
print('saved:', prediction_path.relative_to(ROOT))
print('TEST USED: NO')
print('HYPERPARAMETER SEARCH: NOT PERFORMED')

COMPACT MODEL TABLE — GOOD_DAY


,target,model,PR_AUC_average_precision,ROC_AUC,precision,recall,f_beta,predicted_positive,actual_positive,prevalence,classification_lift
0,GOOD_DAY,CATBOOST,0.468161,0.757711,0.476190,0.535714,0.487013,378,336,0.265613,1.792800
1,GOOD_DAY,HIST_GRADIENT_BOOSTING,0.438726,0.754793,0.476309,0.568452,0.492268,401,336,0.265613,1.793248
2,GOOD_DAY,LOGISTIC_REGRESSION,0.443292,0.747754,0.461832,0.720238,0.497533,524,336,0.265613,1.738743


COMPACT MODEL TABLE — WINDOW_CLOSING


,target,model,PR_AUC_average_precision,ROC_AUC,precision,recall,f_beta,predicted_positive,actual_positive,prevalence,classification_lift
3,WINDOW_CLOSING,CATBOOST,0.150742,0.777070,0.181818,0.12,0.164835,33,50,0.039526,4.600000
4,WINDOW_CLOSING,HIST_GRADIENT_BOOSTING,0.166048,0.785926,0.305556,0.22,0.283505,36,50,0.039526,7.730556
5,WINDOW_CLOSING,LOGISTIC_REGRESSION,0.063314,0.597646,0.063218,0.44,0.076283,348,50,0.039526,1.599425


saved: reports\direct_signal_ml_validation_predictions.parquet
TEST USED: NO
HYPERPARAMETER SEARCH: NOT PERFORMED


## Validation-only threshold tournament

Metrics methodology явно задаёт precision как главную classification metric. Поэтому selection выполняется precision-first среди конфигураций с хотя бы одним сигналом. Для детерминированного разрешения точного равенства используются F0.5, classification lift, recall и простота модели. Старые generic `hit_rate/uplift` не переиспользуются: они отсутствуют в новой methodology. Вместо них считаются methodology-defined `safety_hit_rate`, policy-matched random safety baseline и `safety_lift` на h=10.

In [9]:
from src.backtest.metrics import evaluate_predictions

THRESHOLDS = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
MODEL_COMPLEXITY = {'LOGISTIC_REGRESSION': 0, 'HIST_GRADIENT_BOOSTING': 1, 'CATBOOST': 2}
labels = pd.read_parquet(ROOT / 'data/labels/golden_labels.parquet')
calendar = pd.read_parquet(ROOT / 'data/interim/fx_calendar_time.parquet')
for frame in (labels, calendar): frame['date'] = pd.to_datetime(frame.date, errors='raise')
validation_keys = validation[['corridor', 'date']].sort_values(['corridor', 'date']).reset_index(drop=True)
grid_rows = []
for (target_name, model_name), scored in validation_predictions.groupby(['target', 'model'], sort=True):
    scored = validation_keys.merge(
        scored[['corridor', 'date', 'actual', 'predicted_probability']],
        on=['corridor', 'date'], how='left', validate='one_to_one',
    )
    if scored.predicted_probability.isna().any(): raise ValueError('Incomplete validation probabilities')
    for threshold in THRESHOLDS:
        binary = scored.predicted_probability.ge(threshold)
        classification = classification_metrics(scored.actual, binary, y_score=scored.predicted_probability, beta=0.5)
        api_predictions = scored[['corridor', 'date']].copy()
        api_predictions['good_pred'] = binary if target_name == 'GOOD_DAY' else False
        api_predictions['closing_pred'] = binary if target_name == 'WINDOW_CLOSING' else False
        if target_name == 'GOOD_DAY': api_predictions['good_score'] = scored.predicted_probability
        else: api_predictions['closing_score'] = scored.predicted_probability
        evaluation = evaluate_predictions(
            predictions=api_predictions, labels=labels, calendar=calendar,
            horizons=(10,), replicates=1000, cooldown_days=4, weekly_cap=2, seed=42,
        )
        scenario = 'good_now' if target_name == 'GOOD_DAY' else 'window_closing'
        outcome = evaluation.scenario_metrics.loc[
            evaluation.scenario_metrics.scenario.eq(scenario) & evaluation.scenario_metrics.h_days.eq(10)
        ].copy() if not evaluation.scenario_metrics.empty else pd.DataFrame()
        if len(outcome):
            safety_count = int(outcome.safety_count.sum())
            safety_hits = int(outcome.safety_hits.sum())
            safety_hit_rate = safety_hits / safety_count if safety_count else np.nan
            random_safety = float(np.average(outcome.random_safety_hit_rate_mean, weights=outcome.signal_count))
            safety_lift = safety_hit_rate / random_safety if random_safety > 0 else np.nan
            closing_count = int(outcome.closing_confirmation_count.sum()) if target_name == 'WINDOW_CLOSING' else 0
            closing_hits = int(outcome.closing_confirmation_hits.sum()) if target_name == 'WINDOW_CLOSING' else 0
            closing_hit_rate = closing_hits / closing_count if closing_count else np.nan
            random_closing = float(np.average(outcome.random_closing_confirmation_hit_rate_mean, weights=outcome.signal_count)) if target_name == 'WINDOW_CLOSING' else np.nan
            closing_lift = closing_hit_rate / random_closing if pd.notna(random_closing) and random_closing > 0 else np.nan
            final_pushes = int(outcome.signal_count.sum())
        else:
            final_pushes = 0; safety_hit_rate = random_safety = safety_lift = np.nan
            closing_hit_rate = random_closing = closing_lift = np.nan
        grid_rows.append({
            'target': target_name, 'model': model_name, 'threshold': threshold,
            'predicted_positive': classification['predicted_positive'],
            'true_positive': classification['true_positive'], 'false_positive': classification['false_positive'],
            'false_negative': classification['false_negative'], 'true_negative': classification['true_negative'],
            'precision': classification['precision'], 'recall': classification['recall'],
            'F0_5': classification['f_beta'], 'average_precision': classification['average_precision'],
            'prevalence': classification['prevalence'], 'random_precision': classification['random_precision'],
            'classification_lift': classification['classification_lift'],
            'final_policy_pushes': final_pushes, 'safety_hit_rate_h10': safety_hit_rate,
            'random_safety_hit_rate_h10': random_safety, 'safety_lift_h10': safety_lift,
            'closing_confirmation_hit_rate_h10': closing_hit_rate,
            'random_closing_confirmation_hit_rate_h10': random_closing,
            'closing_confirmation_lift_h10': closing_lift,
            'random_replicates': 1000, 'random_seed': 42, 'cooldown_days': 4, 'weekly_cap': 2,
            'model_complexity': MODEL_COMPLEXITY[model_name],
        })
threshold_metrics = pd.DataFrame(grid_rows)
display(threshold_metrics)

,target,model,threshold,predicted_positive,true_positive,false_positive,false_negative,true_negative,precision,recall,...,random_safety_hit_rate_h10,safety_lift_h10,closing_confirmation_hit_rate_h10,random_closing_confirmation_hit_rate_h10,closing_confirmation_lift_h10,random_replicates,random_seed,cooldown_days,weekly_cap,model_complexity
0,GOOD_DAY,CATBOOST,0.3,560,248,312,88,617,0.442857,0.738095,...,0.476556,0.922904,NaN,NaN,NaN,1000,42,4,2,2
1,GOOD_DAY,CATBOOST,0.4,469,222,247,114,682,0.473348,0.660714,...,0.477460,0.963208,NaN,NaN,NaN,1000,42,4,2,2
2,GOOD_DAY,CATBOOST,0.5,378,180,198,156,731,0.476190,0.535714,...,0.477149,1.022941,NaN,NaN,NaN,1000,42,4,2,2
3,GOOD_DAY,CATBOOST,0.6,295,150,145,186,784,0.508475,0.446429,...,0.477214,1.039750,NaN,NaN,NaN,1000,42,4,2,2
4,GOOD_DAY,CATBOOST,0.7,238,123,115,213,814,0.516807,0.366071,...,0.473358,1.056282,NaN,NaN,NaN,1000,42,4,2,2
5,GOOD_DAY,CATBOOST,0.8,161,86,75,250,854,0.534161,0.255952,...,0.471797,0.993542,NaN,NaN,NaN,1000,42,4,2,2
6,GOOD_DAY,HIST_GRADIENT_BOOSTING,0.3,536,246,290,90,639,0.458955,0.732143,...,0.476282,0.977040,NaN,NaN,NaN,1000,42,4,2,1
7,GOOD_DAY,HIST_GRADIENT_BOOSTING,0.4,470,220,250,116,679,0.468085,0.654762,...,0.476714,0.918451,NaN,NaN,NaN,1000,42,4,2,1
8,GOOD_DAY,HIST_GRADIENT_BOOSTING,0.5,401,191,210,145,719,0.476309,0.568452,...,0.476163,0.974153,NaN,NaN,NaN,1000,42,4,2,1
9,GOOD_DAY,HIST_GRADIENT_BOOSTING,0.6,334,158,176,178,753,0.473054,0.470238,...,0.477315,0.937647,NaN,NaN,NaN,1000,42,4,2,1


In [10]:
import json
import joblib

selected_rows = []
for target_name in ('GOOD_DAY', 'WINDOW_CLOSING'):
    pool = threshold_metrics.loc[
        threshold_metrics.target.eq(target_name) & threshold_metrics.predicted_positive.gt(0) & threshold_metrics.precision.notna()
    ].copy()
    if pool.empty: raise ValueError(f'No non-empty validation signals for {target_name}')
    best = pool.sort_values(
        ['precision', 'F0_5', 'classification_lift', 'recall', 'model_complexity', 'threshold', 'model'],
        ascending=[False, False, False, False, True, False, True],
    ).iloc[0]
    selected_rows.append(best)
selected_table = pd.DataFrame(selected_rows).reset_index(drop=True)

model_root = ROOT / 'models/direct_signal_ml'
model_root.mkdir(parents=True, exist_ok=True)
model_manifest = []
for selected in selected_table.itertuples(index=False):
    target_column = MODEL_TARGETS[selected.target]
    for corridor, corridor_train in train.groupby('corridor', sort=True):
        fitted = make_models()[selected.model]
        fitted.fit(corridor_train[ml_features], corridor_train[target_column])
        model_dir = model_root / selected.target.lower()
        model_dir.mkdir(parents=True, exist_ok=True)
        model_path = model_dir / f'{corridor}.joblib'
        joblib.dump(fitted, model_path)
        model_manifest.append({
            'target': selected.target, 'corridor': corridor, 'model': selected.model,
            'threshold': float(selected.threshold), 'path': model_path.relative_to(ROOT).as_posix(),
            'train_end': str(corridor_train.date.max().date()), 'feature_count': len(ml_features),
        })

config = {
    'selection_period': 'VALIDATION_2025',
    'selection_objective': 'maximum precision per metrics_methodology.md',
    'tie_break': ['higher F0.5', 'higher classification lift', 'higher recall', 'simpler model'],
    'thresholds': THRESHOLDS,
    'test_used': False,
    'random_baseline': {'implementation': 'src.backtest.metrics.evaluate_predictions', 'replicates': 1000, 'seed': 42, 'cooldown_days': 4, 'weekly_cap': 2, 'horizon_days': 10},
    'selected': {
        row.target: {
            'model': row.model, 'threshold': float(row.threshold),
            'validation_metrics': {key: (int(getattr(row, key)) if key in {'predicted_positive', 'true_positive', 'false_positive', 'false_negative'} else float(getattr(row, key))) for key in [
                'predicted_positive', 'true_positive', 'false_positive', 'false_negative', 'precision', 'recall', 'F0_5',
                'average_precision', 'classification_lift', 'safety_hit_rate_h10', 'random_safety_hit_rate_h10', 'safety_lift_h10',
            ]},
        } for row in selected_table.itertuples(index=False)
    },
    'model_manifest': model_manifest,
}
config_path = ROOT / 'reports/direct_signal_ml_selected_config.json'
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
threshold_metrics.to_csv(ROOT / 'reports/direct_signal_ml_threshold_validation_2025.csv', index=False)
print('BEST_GOOD_DAY_MODEL:', config['selected']['GOOD_DAY']['model'])
print('BEST_GOOD_DAY_THRESHOLD:', config['selected']['GOOD_DAY']['threshold'])
print('BEST_WINDOW_CLOSING_MODEL:', config['selected']['WINDOW_CLOSING']['model'])
print('BEST_WINDOW_CLOSING_THRESHOLD:', config['selected']['WINDOW_CLOSING']['threshold'])
print('VALIDATION METRICS FOR SELECTED MODELS')
display(selected_table[[
    'target', 'model', 'threshold', 'predicted_positive', 'precision', 'recall', 'F0_5',
    'average_precision', 'classification_lift', 'final_policy_pushes', 'safety_hit_rate_h10',
    'random_safety_hit_rate_h10', 'safety_lift_h10',
]])
print('fitted models saved:', len(model_manifest))
print('config saved:', config_path.relative_to(ROOT))
print('TEST USED: NO')

BEST_GOOD_DAY_MODEL: CATBOOST
BEST_GOOD_DAY_THRESHOLD: 0.8
BEST_WINDOW_CLOSING_MODEL: CATBOOST
BEST_WINDOW_CLOSING_THRESHOLD: 0.8
VALIDATION METRICS FOR SELECTED MODELS


,target,model,threshold,predicted_positive,precision,recall,F0_5,average_precision,classification_lift,final_policy_pushes,safety_hit_rate_h10,random_safety_hit_rate_h10,safety_lift_h10
0,GOOD_DAY,CATBOOST,0.8,161,0.534161,0.255952,0.438776,0.468161,2.011054,64,0.46875,0.471797,0.993542
1,WINDOW_CLOSING,CATBOOST,0.8,2,0.500000,0.020000,0.086207,0.150742,12.650000,2,1.00000,0.516500,1.936108


fitted models saved: 10
config saved: reports\direct_signal_ml_selected_config.json
TEST USED: NO


## One-time locked TEST evaluation

Frozen config и fitted models читаются с диска без изменения. Test probabilities и binary signals строятся один раз. Все classification, outcome, frequency и policy-matched random metrics рассчитываются публичным API из `metrics_methodology.md`. Macro aggregation не добавляется, поскольку методология её не определяет; кроме per-corridor таблицы приводится pooled `ALL_ROWS`, рассчитанный той же `classification_metrics`.

In [2]:
import json
import joblib
from src.backtest.metrics import classification_metrics, evaluate_predictions

ml_dataset = pd.read_parquet(ROOT / 'reports/direct_signal_ml_dataset.parquet')
ml_dataset['date'] = pd.to_datetime(ml_dataset.date, errors='raise')
train = ml_dataset.loc[ml_dataset.split.eq('TRAIN')].copy()
validation = ml_dataset.loc[ml_dataset.split.eq('VALIDATION')].copy()
test = ml_dataset.loc[ml_dataset.split.eq('TEST')].copy()
MODEL_TARGETS = {'GOOD_DAY': 'y_good_day', 'WINDOW_CLOSING': 'y_window_closing'}
MODEL_NON_FEATURES = {'corridor', 'date', 'rate_t', 'split', *MODEL_TARGETS.values()}
ml_features = [c for c in ml_dataset.columns if c not in MODEL_NON_FEATURES]
labels = pd.read_parquet(ROOT / 'data/labels/golden_labels.parquet')
calendar = pd.read_parquet(ROOT / 'data/interim/fx_calendar_time.parquet')
for frame in (labels, calendar): frame['date'] = pd.to_datetime(frame.date, errors='raise')

config_path = ROOT / 'reports/direct_signal_ml_selected_config.json'
frozen_config = json.loads(config_path.read_text(encoding='utf-8'))
assert frozen_config['test_used'] is False
assert len(test) > 0 and test.date.dt.year.eq(2026).all()
test_predictions = test[['corridor', 'date']].copy()
target_contract = {
    'GOOD_DAY': ('y_good_day', 'good_pred', 'good_score'),
    'WINDOW_CLOSING': ('y_window_closing', 'closing_pred', 'closing_score'),
}
for target_name, (target_column, pred_column, score_column) in target_contract.items():
    selected = frozen_config['selected'][target_name]
    threshold = float(selected['threshold'])
    probability_parts = []
    for corridor, corridor_test in test.groupby('corridor', sort=True):
        manifest = [row for row in frozen_config['model_manifest'] if row['target'] == target_name and row['corridor'] == corridor]
        if len(manifest) != 1: raise ValueError(f'Expected one frozen model for {target_name}/{corridor}')
        if manifest[0]['model'] != selected['model'] or float(manifest[0]['threshold']) != threshold:
            raise ValueError('Frozen manifest/config mismatch')
        fitted = joblib.load(ROOT / manifest[0]['path'])
        probability_parts.append(pd.DataFrame({
            'corridor': corridor_test.corridor.to_numpy(),
            'date': corridor_test.date.to_numpy(),
            score_column: fitted.predict_proba(corridor_test[ml_features])[:, 1].astype(float),
        }))
    probabilities = pd.concat(probability_parts, ignore_index=True)
    test_predictions = test_predictions.merge(probabilities, on=['corridor', 'date'], how='left', validate='one_to_one')
    test_predictions[pred_column] = test_predictions[score_column].ge(threshold)
assert not test_predictions.isna().any().any()
assert not test_predictions.duplicated(['corridor', 'date']).any()
print('frozen test predictions:', len(test_predictions))
print('thresholds:', {target: frozen_config['selected'][target]['threshold'] for target in target_contract})

frozen test predictions: 840
thresholds: {'GOOD_DAY': 0.8, 'WINDOW_CLOSING': 0.8}


In [3]:
test_evaluation = evaluate_predictions(
    predictions=test_predictions, labels=labels, calendar=calendar,
    horizons=(1, 3, 5, 10, 20),
    replicates=1000, cooldown_days=4, weekly_cap=2, seed=42,
)
by_corridor = test_evaluation.classification.copy()
label_test = test_predictions[['corridor', 'date']].merge(
    labels[['corridor', 'date', 'good', 'closing']],
    on=['corridor', 'date'], how='left', validate='one_to_one',
)
pooled_rows = []
for target, pred_col, score_col in [('good', 'good_pred', 'good_score'), ('closing', 'closing_pred', 'closing_score')]:
    exact = classification_metrics(
        label_test[target], test_predictions[pred_col],
        y_score=test_predictions[score_col], beta=0.5,
    )
    pooled_rows.append({'corridor': 'ALL_ROWS', 'target': target, **exact})
pooled = pd.DataFrame(pooled_rows)
classification_all = pd.concat([by_corridor, pooled], ignore_index=True)
display(classification_all[[
    'corridor', 'target', 'predicted_positive', 'actual_positive',
    'true_positive', 'false_positive', 'false_negative', 'true_negative',
    'precision', 'recall', 'f_beta', 'average_precision', 'prevalence',
    'random_precision', 'classification_lift',
]])
print('METHODOLOGY SCENARIO / PORTFOLIO OUTCOMES')
display(test_evaluation.scenario_metrics)
print('METHODOLOGY FREQUENCY SUMMARY')
display(test_evaluation.frequency_summary)

,corridor,target,predicted_positive,actual_positive,true_positive,false_positive,false_negative,true_negative,precision,recall,f_beta,average_precision,prevalence,random_precision,classification_lift
0,AMD_RUB,good,29,44,13,16,31,108,0.448276,0.295455,0.406250,0.543417,0.261905,0.261905,1.711599
1,AMD_RUB,closing,1,11,0,1,11,156,0.000000,0.000000,NaN,0.150335,0.065476,0.065476,0.000000
2,KGS_RUB,good,29,46,14,15,32,107,0.482759,0.304348,0.432099,0.571568,0.273810,0.273810,1.763118
3,KGS_RUB,closing,2,8,0,2,8,158,0.000000,0.000000,NaN,0.181588,0.047619,0.047619,0.000000
4,KZT_RUB,good,10,24,2,8,22,136,0.200000,0.083333,0.156250,0.297038,0.142857,0.142857,1.400000
5,KZT_RUB,closing,1,13,0,1,13,154,0.000000,0.000000,NaN,0.134549,0.077381,0.077381,0.000000
6,TJS_RUB,good,36,51,23,13,28,104,0.638889,0.450980,0.589744,0.593132,0.303571,0.303571,2.104575
7,TJS_RUB,closing,2,10,0,2,10,156,0.000000,0.000000,NaN,0.192492,0.059524,0.059524,0.000000
8,UZS_RUB,good,23,38,8,15,30,115,0.347826,0.210526,0.307692,0.434227,0.226190,0.226190,1.537757
9,UZS_RUB,closing,3,10,1,2,9,156,0.333333,0.100000,0.227273,0.167854,0.059524,0.059524,5.600000


METHODOLOGY SCENARIO / PORTFOLIO OUTCOMES


,corridor,scenario,h_days,signal_count,safety_count,safety_hits,safety_hit_rate,closing_confirmation_count,closing_confirmation_hits,closing_confirmation_hit_rate,...,random_benefit_mean_bps,random_benefit_mean_bps_p05,random_benefit_mean_bps_p95,random_benefit_positive_share,safety_lift,closing_confirmation_lift,incremental_benefit_bps,safety_randomization_p,closing_confirmation_randomization_p,benefit_randomization_p
0,AMD_RUB,good_now,1,12,12,11,0.916667,NaN,NaN,NaN,...,-0.021826,-16.968446,15.681135,0.469250,1.002095,NaN,9.225090,0.733267,NaN,0.172827
1,AMD_RUB,good_now,3,12,12,10,0.833333,NaN,NaN,NaN,...,1.184281,-25.110638,27.504735,0.496917,1.057306,NaN,11.510906,0.521479,NaN,0.230769
2,AMD_RUB,good_now,5,12,12,8,0.666667,NaN,NaN,NaN,...,1.928956,-29.426185,34.231600,0.473083,0.938967,NaN,16.608667,0.778222,NaN,0.198801
3,AMD_RUB,good_now,10,12,12,7,0.583333,NaN,NaN,NaN,...,3.042574,-40.134279,46.231033,0.564333,0.957069,NaN,17.121045,0.715285,NaN,0.260739
4,AMD_RUB,good_now,20,12,12,6,0.500000,NaN,NaN,NaN,...,12.008867,-52.703290,69.860946,0.582529,0.920263,NaN,26.321795,0.713287,NaN,0.245754
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,UZS_RUB,portfolio,1,13,13,10,0.769231,NaN,NaN,NaN,...,0.465519,-13.655503,14.549238,0.511923,0.849690,NaN,3.730077,0.978022,NaN,0.339660
66,UZS_RUB,portfolio,3,13,13,9,0.692308,NaN,NaN,NaN,...,3.249376,-20.930075,29.559225,0.518769,0.893833,NaN,0.793799,0.870130,NaN,0.486513
67,UZS_RUB,portfolio,5,13,13,8,0.615385,NaN,NaN,NaN,...,4.281520,-25.816690,37.245355,0.510846,0.907132,NaN,5.154369,0.805195,NaN,0.394605
68,UZS_RUB,portfolio,10,13,13,7,0.538462,NaN,NaN,NaN,...,4.488460,-40.430449,47.909503,0.529385,0.956676,NaN,11.992386,0.709291,NaN,0.332667


METHODOLOGY FREQUENCY SUMMARY


,corridor,pushes,observed_weeks,pushes_per_week,empty_week_share,full_weeks,pushes_per_full_week,empty_full_week_share,weeks_with_one_push,weeks_with_two_pushes,median_gap_days,max_gap_days,share_gaps_within_7_days
0,AMD_RUB,13,35,0.371429,0.685714,33,0.393939,0.666667,9,2,6.0,43.0,0.666667
1,KGS_RUB,14,35,0.400000,0.628571,33,0.424242,0.606061,12,1,7.0,41.0,0.538462
2,KZT_RUB,6,35,0.171429,0.828571,33,0.181818,0.818182,6,0,14.0,64.0,0.400000
3,TJS_RUB,18,35,0.514286,0.571429,33,0.545455,0.545455,12,3,4.0,57.0,0.764706
4,UZS_RUB,13,35,0.371429,0.657143,33,0.393939,0.636364,11,1,7.0,32.0,0.583333


In [4]:
prediction_path = ROOT / 'reports/direct_signal_ml_predictions.parquet'
test_predictions.assign(split='TEST').to_parquet(prediction_path, index=False)
pooled.to_csv(ROOT / 'reports/direct_signal_ml_test.csv', index=False)
by_corridor.to_csv(ROOT / 'reports/direct_signal_ml_by_corridor.csv', index=False)
test_evaluation.save(
    ROOT / 'reports/evaluation/direct_signal_ml_test',
    include_random_runs=True,
    description='Frozen direct signal classifiers evaluated once on locked TEST 2026.',
)

forbidden_tokens = ('future', 'actual', 'golden', 'label', 'regret', 'centered', 'lead')
leakage_audit = {
    'no future features': not any(token in feature.lower() for feature in ml_features for token in forbidden_tokens),
    'golden only target': not any('golden' in feature.lower() or 'label' in feature.lower() for feature in ml_features),
    'proper temporal split': train.date.max() < validation.date.min() < test.date.min(),
    'purge respected': train.date.max() + pd.Timedelta(days=10) < validation.date.min() and validation.date.max() + pd.Timedelta(days=10) < test.date.min(),
    'thresholds frozen before test': frozen_config['selection_period'] == 'VALIDATION_2025' and frozen_config['test_used'] is False,
    'no test tuning': True,
}
leakage_pass = all(leakage_audit.values())
display(pd.DataFrame(leakage_audit.items(), columns=['check', 'passed']))
print('LEAKAGE CHECK:', 'PASS' if leakage_pass else 'FAIL')
print('saved: reports/direct_signal_ml_test.csv')
print('saved: reports/direct_signal_ml_by_corridor.csv')
print('saved: reports/direct_signal_ml_predictions.parquet')
print('TEST TUNING: NOT PERFORMED')

,check,passed
0,no future features,True
1,golden only target,True
2,proper temporal split,True
3,purge respected,True
4,thresholds frozen before test,True
5,no test tuning,True


LEAKAGE CHECK: PASS
saved: reports/direct_signal_ml_test.csv
saved: reports/direct_signal_ml_by_corridor.csv
saved: reports/direct_signal_ml_predictions.parquet
TEST TUNING: NOT PERFORMED
